#The other Three Remaining Rings

In [ ]:
from fractions import Fraction as Fr
import math, itertools

# =====================================================================
# Ring of integers O_d of Q(sqrt(d)), d a negative squarefree integer.
# Element (x,y) means:
#   d ≡ 2,3 (mod 4):  x + y*sqrt(d),         conj=(x,-y),  N=x^2+|d|*y^2
#   d ≡ 1 (mod 4):    x + y*(1+sqrt(d))/2,   conj=(x+y,-y),N=x^2+x*y+((1-d)/4)*y^2
# =====================================================================
class Ring:
    def __init__(self, d):
        self.d = d
        self.half = (d % 4 == 1)          # use the (1+sqrt d)/2 integral basis
        self.k = (1 - d)//4 if self.half else None
    def conj(self, z):
        x, y = z
        return (x + y, -y) if self.half else (x, -y)
    def N(self, z):
        x, y = z
        if self.half:
            return x*x + x*y + self.k*y*y
        return x*x + (-self.d)*y*y
    def mul(self, a, b):
        x, y = a; u, v = b
        if self.half:
            # (x + y w)(u + v w), w=(1+sqrt d)/2, w^2 = w - k  (k=(1-d)/4)
            return (x*u - self.k*y*v, x*v + y*u + y*v)
        # (x + y t)(u + v t), t=sqrt d, t^2 = d
        return (x*u + self.d*y*v, x*v + y*u)
    def to_complex(self, z):
        x, y = z
        if self.half:
            return complex(x + y*0.5, y*math.sqrt(-self.d)/2)
        return complex(x, y*math.sqrt(-self.d))

RINGS = {d: Ring(d) for d in (-1, -2, -3, -7, -11)}
MU2 = {-1: Fr(1,2), -2: Fr(3,4), -3: Fr(1,3), -7: Fr(4,7), -11: Fr(9,11)}  # covering radius^2

def lattice_points(R, Bnorm):
    """All z in O_d with N(z) <= Bnorm."""
    out = []
    M = int(math.isqrt(int(Bnorm))) + 2
    rng = range(-M-abs(R.d), M+abs(R.d)+1)
    for x in rng:
        for y in rng:
            if R.N((x, y)) <= Bnorm:
                out.append((x, y))
    return out

# =====================================================================
# PART A : 2x2 Hermitian forms  X = [[a, b],[conj b, c]], a,c in Z, b in O.
#          Represent X as (a, c, b).
# =====================================================================
def re(R, z):                      # 2*Re(z) = Tr(z) = z + conj(z), an integer
    s = (z[0] + R.conj(z)[0], z[1] + R.conj(z)[1])
    assert s[1] == 0
    return s[0]

def det2(R, X):
    a, c, b = X
    return a*c - R.N(b)

def adj2(R, X):                    # adjugate of [[a,b],[b*,c]] is [[c,-b],[-b*,a]]
    a, c, b = X
    return (c, a, (-b[0], -b[1]))

def vXv2(R, X, v):                 # v=(p,q):  a N(p)+c N(q) + 2Re(conj(p) b q)
    a, c, b = X
    p, q = v
    return a*R.N(p) + c*R.N(q) + re(R, R.mul(R.mul(R.conj(p), b), q))

def is_psd2(R, X):
    a, c, b = X
    return a >= 0 and c >= 0 and det2(R, X) >= 0

def min2(R, X, vecs):              # minimum of vXv over O^2 \ 0
    best = None
    for p in vecs:
        for q in vecs:
            if p == (0, 0) and q == (0, 0):
                continue
            val = vXv2(R, X, (p, q))
            if val > 0 and (best is None or val < best):
                best = val
    return best

def has_rank1_below(R, X, vecs):
    """Exists primitive v with X - v v^* PSD ?  (direct definition of 'not stuck')."""
    a, c, b = X
    for p in vecs:
        for q in vecs:
            if p == (0, 0) and q == (0, 0):
                continue
            # X - vv^*: a-N(p), c-N(q), b - p*conj(q)
            pcq = R.mul(p, R.conj(q))
            Y = (a - R.N(p), c - R.N(q), (b[0]-pcq[0], b[1]-pcq[1]))
            if is_psd2(R, Y):
                return True
    return False

def is_stuck2_via_adj(R, X, vecs):
    """Criterion: positive-definite X is stuck iff m(adj X) > det X."""
    return min2(R, adj2(R, X), vecs) > det2(R, X)

# ---- Minkowski reduction of a binary Hermitian form, valid over any O_d (PID) ----
def reduce_b_mod_a(R, b, a):
    """Reduce b modulo a*O_d to the nearest lattice point (a>0 integer)."""
    # b/a as a complex number, find nearest lattice element q, return b - a*q.
    w = R.to_complex(b) / a
    if R.half:
        ya = 2*w.imag/math.sqrt(-R.d); xa = w.real - ya/2
    else:
        xa = w.real; ya = w.imag/math.sqrt(-R.d)
    best = None; bestN = None
    for xx in (math.floor(xa), math.ceil(xa)):
        for yy in (math.floor(ya), math.ceil(ya)):
            q = (xx, yy)
            r = (b[0]-a*q[0], b[1]-a*q[1])
            if bestN is None or R.N(r) < bestN:
                bestN = R.N(r); best = r
    return best

def canonical2(R, X):
    """Canonical representative of the GL2(O)-orbit (Minkowski reduction)."""
    a, c, b = X
    for _ in range(200):
        b = reduce_b_mod_a(R, b, a) if a > 0 else b
        if c < a:                      # swap basis vectors: [[a,b],[b*,c]]->[[c,b*],[b,a]]
            a, c, b = c, a, R.conj(b)
            continue
        break
    # normalise b by a unit so its representative is canonical (use (N(b), sorted coords))
    return (a, c, (abs(b[0])+abs(b[1]),))  # invariant signature: (a, c, "shape" of b via N)

def orbit_key(R, X):
    a, c, b = X
    bb = reduce_b_mod_a(R, b, a) if a > 0 else b
    a2, c2 = (a, c)
    if c2 < a2:
        a2, c2 = c2, a2
    return (a2, c2, R.N(bb))

def search_2x2(R, d, a_max, c_max):
    """Find all stuck 2x2 forms (no rank-one below) within a generous reduced box."""
    mu2 = MU2[d]
    vecs = lattice_points(R, max(a_max, c_max) + 4)           # for min / rank-1 search
    stuck = []
    for a in range(1, a_max+1):
        # reduced: N(b) <= mu2 * a^2  (covering-radius reduction); also PSD needs N(b)<=a*c
        bnorm_cap = int(mu2 * a * a) + 1
        bvs = lattice_points(R, bnorm_cap)
        for c in range(a, c_max+1):
            for b in bvs:
                if R.N(b) > a*c:
                    continue
                X = (a, c, b)
                if det2(R, X) < 1:           # positive definite only
                    continue
                if not has_rank1_below(R, X, vecs):
                    stuck.append(X)
    # group into orbits
    seen = {}
    for X in stuck:
        seen.setdefault(orbit_key(R, X), X)
    return stuck, list(seen.values()), vecs

def three_is_norm(R):
    for x in range(-3, 4):
        for y in range(-3, 4):
            if R.N((x, y)) == 3:
                return (x, y)
    return None

print("="*72)
print("PART A.  2x2 Hermitian PSD cone over the five norm-Euclidean rings")
print("="*72)
NAMES = {-1:"Z[i]", -2:"Z[sqrt(-2)]", -3:"Z[omega]", -7:"Z[(1+sqrt-7)/2]", -11:"Z[(1+sqrt-11)/2]"}
results = {}
for d in (-1, -2, -3, -7, -11):
    R = RINGS[d]
    stuck, orbits, vecs = search_2x2(R, d, a_max=6, c_max=14)
    mu2 = MU2[d]
    fourmu2_ge3 = (4*mu2 >= 3)
    nrm3 = three_is_norm(R)
    # cross-check criterion vs direct test on all reduced PD forms in the box
    mism = 0; checked = 0
    for a in range(1, 7):
        for c in range(a, 9):
            for b in lattice_points(R, int(mu2*a*a)+1):
                if R.N(b) > a*c: continue
                X=(a,c,b)
                if det2(R,X) < 1: continue
                checked += 1
                if is_stuck2_via_adj(R, X, vecs) != (not has_rank1_below(R, X, vecs)):
                    mism += 1
    results[d] = dict(stuck=stuck, orbits=orbits, mu2=mu2,
                      fourmu2_ge3=fourmu2_ge3, nrm3=nrm3, mism=mism, checked=checked)
    print(f"\n  d={d:>3}  {NAMES[d]}")
    print(f"     mu^2 = {mu2}   (4 mu^2 >= 3 ? {fourmu2_ge3})    3 a norm ? "
          f"{'yes '+str(nrm3) if nrm3 else 'no (inert)'}")
    print(f"     stuck forms found: {len(stuck)};  distinct GL2(O)-orbits: {len(orbits)}")
    for X in orbits:
        a,c,b = X
        print(f"        sporadic root: [[{a}, {b}],[conj, {c}]]  det={det2(R,X)}  "
              f"min={min2(R,X,vecs)}")
    print(f"     criterion (m(adj X)>det X) vs direct rank-one test: "
          f"{mism} mismatch / {checked} forms")

# Explicit checks on X0 (d=-2) and X1 (d=-11)
print("\n  --- explicit checks on the named sporadic roots ---")
R2 = RINGS[-2];  X0 = (2, 2, (1, 1))          # [[2, 1+sqrt(-2)],[1-sqrt(-2),2]]
R11 = RINGS[-11]; X1 = (2, 2, (0, 1))         # b = tau = (1+sqrt(-11))/2  -> (x,y)=(0,1)
for tag, R, X in [("X0 over Z[sqrt-2]", R2, X0), ("X1 over Z[(1+sqrt-11)/2]", R11, X1)]:
    vecs = lattice_points(R, 12)
    print(f"     {tag}:  det={det2(R,X)}  min={min2(R,X,vecs)}  "
          f"stuck(no rank-1 below)={not has_rank1_below(R,X,vecs)}  "
          f"PD={det2(R,X)>0 and X[0]>0}")

# =====================================================================
# PART B : 3x3 Hermitian forms over Z[i] (d=-1) and Z[omega] (d=-3).
#   X = [[a, b, c],[b*, d, e],[c*, e*, f]],  a,d,f in Z, b,c,e in O.
#   Represent X as (a, d, f, b, c, e).
# =====================================================================
def tr_re(R, z):                          # 2 Re(z) = Tr(z)
    s = (z[0]+R.conj(z)[0], z[1]+R.conj(z)[1]); assert s[1]==0; return s[0]

def vXv3(R, X, p, q, r):
    a,d,f,b,c,e = X
    return (a*R.N(p)+d*R.N(q)+f*R.N(r)
            + tr_re(R, R.mul(R.mul(R.conj(p),b),q))
            + tr_re(R, R.mul(R.mul(R.conj(p),c),r))
            + tr_re(R, R.mul(R.mul(R.conj(q),e),r)))

def det3(R, X):
    a,d,f,b,c,e = X
    bs,cs,es = R.conj(b),R.conj(c),R.conj(e)
    term_a = a*(d*f - R.N(e))
    ecs = R.mul(e,cs)
    m1 = (bs[0]*f-ecs[0], bs[1]*f-ecs[1]); term_b = R.mul(b,m1)
    bse = R.mul(bs,es); m2 = (bse[0]-d*cs[0], bse[1]-d*cs[1]); term_c = R.mul(c,m2)
    val = (term_a - term_b[0] + term_c[0], -term_b[1] + term_c[1])
    return val                            # (real, imag); imag must be 0

def adj3(R, X):
    a,d,f,b,c,e = X
    bs,cs,es = R.conj(b),R.conj(c),R.conj(e)
    A11=d*f-R.N(e); A22=a*f-R.N(c); A33=a*d-R.N(b)
    A12=(-(b[0]*f-R.mul(c,es)[0]), -(b[1]*f-R.mul(c,es)[1]))
    be=R.mul(b,e); A13=(be[0]-c[0]*d, be[1]-c[1]*d)
    cbs=R.mul(c,bs); A23=(-(a*e[0]-cbs[0]), -(a*e[1]-cbs[1]))
    return (A11,A22,A33,A12,A13,A23)

def is_psd3(R, X):
    a,d,f,b,c,e = X
    if a<0 or d<0 or f<0: return False
    if a*d-R.N(b)<0 or a*f-R.N(c)<0 or d*f-R.N(e)<0: return False
    de = det3(R,X); return de[1]==0 and de[0]>=0

def search_3x3(R, d, detmax, off_full=True, TMAX=22):
    offpts = [(x, y) for x in range(-8, 9) for y in range(-8, 9)]   # off-diagonal box
    vpts = [(x, y) for x in range(-6, 7) for y in range(-6, 7)]     # vectors for stuck test
    vecs = [(R.N(p)+R.N(q)+R.N(r), p, q, r)
            for p in vpts for q in vpts for r in vpts
            if not (p==(0,0) and q==(0,0) and r==(0,0)) and R.N(p)+R.N(q)+R.N(r) <= TMAX]
    vecs.sort(key=lambda z: z[0])
    def stuck(X):
        D = det3(R,X)[0]
        if D <= 0: return False
        A = adj3(R,X); a,dd,f,_,_,_ = X; bound = a+dd+f      # >= lambda_max(X)
        for (t,p,q,r) in vecs:
            if t > bound: break
            if 0 < vXv3(R,A,p,q,r) <= D: return False         # rank-one below -> not stuck
        return True
    def offd(diag):
        h = diag if off_full else diag//2
        return [z for z in offpts if abs(z[0]) <= h and abs(z[1]) <= h]
    cnt = 0; stuck_forms = []
    for a in range(1, 3):                              # minimum diagonal <= 2
        for dd in range(a, 9):
            for f in range(dd, 13):
                for b in offd(a):
                    if R.N(b) > a*dd: continue
                    for c in offd(a):
                        if R.N(c) > a*f: continue
                        for e in offd(dd):
                            if R.N(e) > dd*f: continue
                            X = (a,dd,f,b,c,e); De = det3(R,X)
                            if De[1] != 0: continue
                            D = De[0]
                            if D < 1 or D > detmax or not is_psd3(R,X): continue
                            cnt += 1
                            if stuck(X): stuck_forms.append((X,D))
    return cnt, stuck_forms

print()
print("="*72)
print("PART B.  3x3 Hermitian PSD cone over Z[i] and Z[omega]")
print("="*72)
# Hermite-Humbert: stuck => det < gamma_6^3 (|d_K|/4)^{3/2}
import math as _m
g6c = (64/3)**0.5
for d, dK in [(-1,4), (-3,3)]:
    bound = g6c*((dK/4)**1.5)
    detmax = _m.floor(bound - 1e-9)
    R = RINGS[d]
    cnt, stuck_forms = search_3x3(R, d, detmax)
    print(f"\n  d={d:>3}  {NAMES[d]}")
    print(f"     stuck => det < {bound:.4f}  => det in 1..{detmax}")
    print(f"     reduced PSD forms scanned (det<={detmax}): {cnt}")
    print(f"     STUCK ternary forms found: {len(stuck_forms)}  "
          f"=> single root e1e1* {'GENERATES (no sporadic root)' if not stuck_forms else 'FAILS'}")

print()
print("="*72)
print("SUMMARY  (number of roots = e1e1* plus sporadic roots)")
print("="*72)
print("  rank 2:  Z[i] {e1e1*} | Z[w] {e1e1*} | Z[(1+s-7)/2] {e1e1*}"
      " | Z[s-2] {e1e1*,X0} | Z[(1+s-11)/2] {e1e1*,X1}")
print("  rank 3:  Z[i] {e1e1*} | Z[w] {e1e1*}   (single root; no stuck form)")
print("  X0 = [[2,1+sqrt(-2)],[1-sqrt(-2),2]],  X1 = [[2,(1+sqrt(-11))/2],[.,2]]; both det 1, min 2")


# SOC (n=11)

In [ ]:
import itertools
import math
import time
from math import gcd
import numba
from numba import njit

@njit(fastmath=True, cache=True)
def is_primitive(v):
    """GCD of all components == 1"""
    g = abs(v[0])
    for i in range(1, len(v)):
        g = math.gcd(g, abs(v[i]))
        if g == 1:
            return True
    return g == 1


def enumerate_short_roots_with_vectors(max_coord=4, d=10):
    """Enumerate primitive norm-+2 vectors (short roots) with concrete signed representatives"""
    start = time.time()
    roots = []
    seen = set()
    coord_range = range(-max_coord, max_coord + 1)

    total_candidates = len(coord_range) ** d
    print(f"Starting short-root enumeration (max_coord={max_coord}, ~{total_candidates:,} candidates)...")

    last_print = time.time()
    count = 0

    for coords10 in itertools.product(coord_range, repeat=d):
        count += 1

        if time.time() - last_print > 8 or count % 1_000_000 == 0:
            elapsed = time.time() - start
            print(f"  Progress: {count:,}/{total_candidates:,} checked | Elapsed: {elapsed:.1f}s")
            last_print = time.time()

        s = sum(x * x for x in coords10)
        target_sq = s - 2
        if target_sq < 0:
            continue

        x11 = math.isqrt(target_sq)
        if x11 * x11 == target_sq and x11 <= max_coord:
            for sign in [1, -1] if x11 > 0 else [1]:
                v_list = list(coords10) + [sign * x11]
                if is_primitive(v_list):
                    t = tuple(sorted(abs(x) for x in v_list))
                    if t not in seen:
                        seen.add(t)
                        roots.append(tuple(v_list))

    # Sort by max abs coordinate then lexicographically
    roots.sort(key=lambda v: (max(abs(x) for x in v), v))
    elapsed = time.time() - start
    print(f"\n✅ Short roots finished in {elapsed:.2f} seconds")
    print(f"Found {len(roots)} distinct primitive short roots (norm +2)")
    return roots


def enumerate_primitive_isotropic_with_vectors(max_coord=4, d=10):
    """Enumerate primitive isotropic vectors q(v)=0 with concrete signed representatives"""
    start = time.time()
    isos = []                     # list of concrete signed tuples
    seen = set()
    coord_range = range(-max_coord, max_coord + 1)

    total_candidates = len(coord_range) ** d
    print(f"Starting isotropic enumeration (max_coord={max_coord}, ~{total_candidates:,} candidates)...")

    last_print = time.time()
    count = 0

    for coords10 in itertools.product(coord_range, repeat=d):
        count += 1

        if time.time() - last_print > 8 or count % 1_000_000 == 0:
            elapsed = time.time() - start
            print(f"  Progress: {count:,}/{total_candidates:,} checked | Elapsed: {elapsed:.1f}s")
            last_print = time.time()

        s = sum(x * x for x in coords10)
        x11 = math.isqrt(s)
        if x11 * x11 == s and x11 <= max_coord:
            for sign in [1, -1] if x11 > 0 else [1]:
                v_list = list(coords10) + [sign * x11]
                if any(x != 0 for x in v_list) and is_primitive(v_list):
                    t = tuple(sorted(abs(x) for x in v_list))
                    if t not in seen:
                        seen.add(t)
                        isos.append(tuple(v_list))

    # Sort by max abs coordinate then lexicographically
    isos.sort(key=lambda v: (max(abs(x) for x in v), v))
    elapsed = time.time() - start
    print(f"\n✅ Isotropic vectors finished in {elapsed:.2f} seconds")
    print(f"Found {len(isos)} distinct primitive isotropic vectors (q=0)")
    return isos


if __name__ == "__main__":
    print("="*80)
    print("SOC(11) / I_{10,1}  —  Primitive vectors enumeration (concrete signed reps)")
    print("="*80)

    # You can change max_coord here (3 is fast, 4 is ~1 hour)
    MAX_COORD = 3

    print("\n[1/2] Enumerating short roots (norm +2)...")
    short_roots = enumerate_short_roots_with_vectors(max_coord=MAX_COORD)

    print("\nConcrete short-root vectors (first 20 shown):")
    for v in short_roots[:20]:
        print(v)
    if len(short_roots) > 20:
        print(f"... + {len(short_roots)-20} more")

    print("\n[2/2] Enumerating primitive isotropic vectors (q=0)...")
    isos = enumerate_primitive_isotropic_with_vectors(max_coord=MAX_COORD)

    print("\nConcrete isotropic vectors (all shown):")
    for v in isos:
        print(v)

    print("\n" + "="*80)
    print("✅ DONE! You now have concrete signed representatives for both sets.")
    print("Copy the two lists (short_roots and isos) for the next step (Vinberg reduction).")

In [ ]:
import numpy as np
from collections import deque
import time

# ================== YOUR SHORT ROOTS (max_coord=4) ==================
short_roots = [
    (-1, -1, -1, 0, 0, 0, 0, 0, 0, 0, 1),
    (-1, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0),
    (-2, -1, -1, 0, 0, 0, 0, 0, 0, 0, 2),
    (-1, -1, -1, -1, -1, -1, 0, 0, 0, 0, 2),
    (-3, -1, -1, 0, 0, 0, 0, 0, 0, 0, 3),
    (-2, -2, -1, -1, -1, 0, 0, 0, 0, 0, 3),
    (-2, -1, -1, -1, -1, -1, -1, -1, 0, 0, 3),
    (-4, -1, -1, 0, 0, 0, 0, 0, 0, 0, 4),
    (-3, -3, 0, 0, 0, 0, 0, 0, 0, 0, 4),
    (-3, -2, -2, -1, 0, 0, 0, 0, 0, 0, 4),
    (-3, -2, -1, -1, -1, -1, -1, 0, 0, 0, 4),
    (-3, -1, -1, -1, -1, -1, -1, -1, -1, -1, 4),
    (-2, -2, -2, -2, -1, -1, 0, 0, 0, 0, 4),
    (-2, -2, -2, -1, -1, -1, -1, -1, -1, 0, 4)
]

# ================== LORENTZ INNER PRODUCT & REFLECTION ==================
def lorentz_dot(v, w):
    """<v, w> = v1 w1 + ... + v10 w10  -  v11 w11"""
    return sum(a * b for a, b in zip(v, w[:10])) - v[10] * w[10]

def reflect(v, r):
    """s_r(v) = v - 2 * (v·r)/ (r·r) * r   (since ||r||^2 = 2)"""
    b = lorentz_dot(v, r)
    return tuple(int(x - 2 * b * y) for x, y in zip(v, r))

def norm2(v):
    """Quadratic form q(v) = ||v||^2"""
    return sum(x*x for x in v[:10]) - v[10]*v[10]

def is_primitive(v):
    g = abs(v[0])
    for x in v[1:]:
        g = math.gcd(g, abs(x))
        if g == 1:
            return True
    return g == 1

# ================== VINBERG CHAMBER-WALKING ALGORITHM ==================
def vinberg_chamber_walk(initial_roots, max_height=6, max_new_roots=1000):
    """
    Full Vinberg algorithm: discovers new simple roots by walking adjacent chambers.
    Returns list of all discovered simple roots (up to the height bound).
    """
    start_time = time.time()

    # Canonical representation for deduplication: sorted absolute coordinates
    known_simple = set(tuple(sorted(abs(x) for x in r)) for r in initial_roots)
    simple_roots = list(initial_roots)          # concrete signed vectors
    queue = deque(simple_roots)                 # roots to explore reflections from

    new_roots_found = 0
    iteration = 0

    print(f"Starting Vinberg chamber-walking with {len(simple_roots)} initial simple roots...")
    print(f"Target height bound: {max_height} | Max new roots: {max_new_roots}\n")

    while queue and new_roots_found < max_new_roots:
        iteration += 1
        r = queue.popleft()

        # Reflect over every known simple root to explore adjacent chambers
        for s in simple_roots:
            if lorentz_dot(r, s) >= 0:
                continue  # not adjacent

            # Reflect the current root over s to get a candidate for a new wall
            candidate = reflect(r, s)

            # Check if it is a new primitive norm-+2 vector
            if norm2(candidate) == 2 and is_primitive(candidate):
                canon = tuple(sorted(abs(x) for x in candidate))
                if canon not in known_simple:
                    known_simple.add(canon)
                    simple_roots.append(candidate)
                    queue.append(candidate)
                    new_roots_found += 1

                    max_c = max(abs(x) for x in candidate)
                    print(f"  → NEW simple root #{new_roots_found:3d} (height {max_c}) : {candidate}")

        if iteration % 100 == 0:
            elapsed = time.time() - start_time
            print(f"  Progress: {iteration} iterations | {len(simple_roots)} simple roots | Elapsed: {elapsed:.1f}s")

    elapsed = time.time() - start_time
    print(f"\n✅ Vinberg chamber-walking finished in {elapsed:.1f} seconds")
    print(f"   Discovered {len(simple_roots)} simple roots total ({new_roots_found} new)")
    print(f"   Max height reached in new roots: {max((max(abs(x) for x in r) for r in simple_roots), default=0)}")

    return simple_roots


# ================== RUN THE ALGORITHM ==================
if __name__ == "__main__":
    print("=" * 90)
    print("VINBERG CHAMBER-WALKING FOR I_{10,1}  (SOC(11) reflection group)")
    print("=" * 90)

    all_simple_roots = vinberg_chamber_walk(short_roots, max_height=6, max_new_roots=2000)

    print("\nFirst 20 discovered simple roots (sorted by max coordinate):")
    all_simple_roots.sort(key=lambda v: (max(abs(x) for x in v), v))
    for r in all_simple_roots[:20]:
        print(r)

    print(f"\nTotal simple roots found: {len(all_simple_roots)}")
    print("If the number stabilizes quickly → finite covolume → (R,G)-finite generation likely holds.")
    print("If it keeps growing indefinitely → infinite covolume → NOT (R,G)-finitely generated.")

# Hermitian PSD Cone Z[i]

In [ ]:
import random, time

# ---------- Gaussian integer operations----------
def g_add(u,v):  return (u[0]+v[0], u[1]+v[1])
def g_sub(u,v):  return (u[0]-v[0], u[1]-v[1])
def g_mul(u,v):  return (u[0]*v[0] - u[1]*v[1], u[0]*v[1] + u[1]*v[0])
def g_conj(u):   return (u[0], -u[1])
def g_norm(u):   return u[0]*u[0] + u[1]*u[1]
def g_neg(u):    return (-u[0], -u[1])

def g_div(u, v):
    num = g_mul(u, g_conj(v))
    den = g_norm(v)
    return (round(num[0] / den), round(num[1] / den))

# ---------- Reduction ----------
def reduce_form(a, b, c):
    a_int, c_int = a, c
    a_tup = (a_int, 0)
    c_tup = (c_int, 0)
    b = b
    U00, U01 = (1,0), (0,0)
    U10, U11 = (0,0), (1,0)

    while True:
        if a_int > c_int:
            a_int, c_int = c_int, a_int
            a_tup, c_tup = c_tup, a_tup
            b = g_conj(b)
            U00, U01, U10, U11 = U01, U00, U11, U10
            continue
        if a_int == 0 or 2 * g_norm(b) <= a_int * a_int:
            break
        t = g_div(b, a_tup)
        if t == (0,0):
            break
        b_old = b
        b = g_sub(b, g_mul(t, a_tup))
        term1 = g_mul(t, g_conj(b_old))
        term2 = g_mul(g_conj(t), b_old)
        term3 = g_mul((g_norm(t),0), a_tup)
        c_tup = g_sub(c_tup, term1)
        c_tup = g_sub(c_tup, term2)
        c_tup = g_add(c_tup, term3)
        c_int = c_tup[0]
        # update U = U * V
        U01 = g_sub(U01, g_mul(t, U00))
        U11 = g_sub(U11, g_mul(t, U10))
    return a_int, b, c_int, (U00, U01, U10, U11)

# ---------- Inverse of unimodular matrix ----------
def inverse_unimodular(U):
    U00, U01, U10, U11 = U
    det = g_sub(g_mul(U00, U11), g_mul(U01, U10))
    det_inv = g_conj(det)           # because det is unit (±1, ±i)
    inv00 = g_mul(det_inv, U11)
    inv01 = g_mul(g_neg(det_inv), U01)
    inv10 = g_mul(g_neg(det_inv), U10)
    inv11 = g_mul(det_inv, U00)
    return (inv00, inv01, inv10, inv11)

# ---------- Decomposition ----------
def decompose_form(a_in, b_in, c_in):
    a, c = a_in, c_in
    b = b_in
    comps = []
    while not (a == 0 and b == (0,0) and c == 0):
        a_red, b_red, c_red, U = reduce_form(a, b, c)
        U_inv = inverse_unimodular(U)
        if a_red == 0:
            k = c_red
            if k == 0:
                break
            # v = (U^{-1})^* e2 = conj(second row of U^{-1})
            v = (g_conj(U_inv[2]), g_conj(U_inv[3]))
            comps.extend([v] * k)
            vp_norm = g_norm(v[0]); vq_norm = g_norm(v[1])
            cross = g_mul(v[0], g_conj(v[1]))
            a -= k * vp_norm
            b = (b[0] - k * cross[0], b[1] - k * cross[1])
            c -= k * vq_norm
        else:
            # v = (U^{-1})^* e1 = conj(first row of U^{-1})
            v = (g_conj(U_inv[0]), g_conj(U_inv[1]))
            comps.append(v)
            vp_norm = g_norm(v[0]); vq_norm = g_norm(v[1])
            cross = g_mul(v[0], g_conj(v[1]))
            a -= vp_norm
            b = (b[0] - cross[0], b[1] - cross[1])
            c -= vq_norm
    return comps

# ---------- Check functions ----------
def check_decomposition(a0, b0, c0, comps):
    ra, rc = 0, 0
    rb = (0,0)
    for p,q in comps:
        ra += g_norm(p)
        rc += g_norm(q)
        cross = g_mul(p, g_conj(q))
        rb = (rb[0] + cross[0], rb[1] + cross[1])
    return (ra == a0 and rb == b0 and rc == c0)

def is_primitive_vec(p, q):
    u, v = p, q
    while v != (0,0):
        r = g_sub(u, g_mul(g_div(u, v), v))
        u, v = v, r
    return g_norm(u) == 1

# ---------- Random matrix generation (PSD by construction) ----------
def random_psd(max_coeff=5, nsum=4):
    a, c = 0, 0
    b = (0,0)
    for _ in range(random.randint(1, nsum)):
        p = (random.randint(-max_coeff, max_coeff), random.randint(-max_coeff, max_coeff))
        q = (random.randint(-max_coeff, max_coeff), random.randint(-max_coeff, max_coeff))
        a += g_norm(p)
        c += g_norm(q)
        cross = g_mul(p, g_conj(q))
        b = (b[0] + cross[0], b[1] + cross[1])
    return a, b, c

# ---------- Test ----------
if __name__ == "__main__":
    NUM_TESTS = 1_000_000
    random.seed(12345)
    print(f"Running {NUM_TESTS:,} random tests...")
    start = time.time()
    failures = 0
    for i in range(NUM_TESTS):
        a, b, c = random_psd(10, 5)
        try:
            comp = decompose_form(a, b, c)
            if not check_decomposition(a, b, c, comp):
                print(f"FAIL test {i}: reconstruction mismatch")
                failures += 1
                break
            # primitivity check
            if not all(is_primitive_vec(p,q) for p,q in comp):
                 print(f"FAIL test {i}: non-primitive vector")
                 failures += 1
                 break
        except Exception as e:
            print(f"FAIL test {i}: exception {e}")
            failures += 1
            break
        if i % 100_000 == 99_999:
            elapsed = time.time() - start
            print(f"  {i+1:,} tests done ({elapsed:.1f}s)")
    total_time = time.time() - start
    if failures == 0:
        print(f"All {NUM_TESTS:,} tests passed in {total_time:.1f} seconds.")
    else:
        print(f"{failures} failures detected.")

# Hermitian PSD Cone Eisenstein Z[ω]

In [ ]:
# Exact integer arithmetic for Eisenstein integers Z[ω]
# ω = (-1 + i√3)/2, ω^2 + ω + 1 = 0
# Represent a + bω as (a,b)

import random, time, sys
import math

# ---------- Eisenstein arithmetic ----------
def w_add(u,v):    return (u[0]+v[0], u[1]+v[1])
def w_sub(u,v):    return (u[0]-v[0], u[1]-v[1])
def w_mul(u,v):
    # (a + bω)(c + dω) = (ac - bd) + (ad + bc - bd) ω
    a,b = u
    c,d = v
    real = a*c - b*d
    imag = a*d + b*c - b*d
    return (real, imag)

def w_conj(u):
    # conj(a + bω) = (a - b) - b ω   (since conj(ω) = ω^2 = -1 - ω)
    a,b = u
    return (a - b, -b)

def w_norm(u):
    a,b = u
    return a*a - a*b + b*b

def w_neg(u):
    return (-u[0], -u[1])


def w_div(u, v):
    """Return t in Z[ω] nearest to u/v (exact complex distance)."""
    num = w_mul(u, w_conj(v))
    den = w_norm(v)
    a_prime, b_prime = num
    # Cartesian coordinates of num/den
    x = (a_prime - b_prime / 2.0) / den
    y = (b_prime * math.sqrt(3) / 2.0) / den
    # round to nearest Eisenstein integer
    b = round(2 * y / math.sqrt(3))
    a = round(x + b / 2.0)
    return (a, b)


# ---------- Reduction (Eisenstein) ----------
def reduce_form(a, b, c):
    a_int, c_int = a, c
    a_tup = (a_int, 0)   # as Eisenstein integer (a + 0ω)
    c_tup = (c_int, 0)
    b = b
    U00, U01 = (1,0), (0,0)
    U10, U11 = (0,0), (1,0)

    while a_int > c_int or 3 * w_norm(b) > a_int * a_int:
        if a_int > c_int:
            a_int, c_int = c_int, a_int
            a_tup, c_tup = c_tup, a_tup
            b = w_conj(b)
            U00, U01, U10, U11 = U01, U00, U11, U10
            continue
        if a_int == 0 or 3 * w_norm(b) <= a_int * a_int:
            break
        t = w_div(b, (a_int, 0))
        if t == (0,0):
            break
        b_old = b
        b = w_sub(b, w_mul(t, a_tup))
        term1 = w_mul(t, w_conj(b_old))
        term2 = w_mul(w_conj(t), b_old)
        term3 = w_mul((w_norm(t), 0), a_tup)
        c_tup = w_sub(c_tup, term1)
        c_tup = w_sub(c_tup, term2)
        c_tup = w_add(c_tup, term3)
        c_int = c_tup[0]   # real part
        # update U = U * V
        U01 = w_sub(U01, w_mul(t, U00))
        U11 = w_sub(U11, w_mul(t, U10))
    return a_int, b, c_int, (U00, U01, U10, U11)

# ---------- Inverse of unimodular matrix (over Z[ω]) ----------
def inverse_unimodular(U):
    U00, U01, U10, U11 = U
    det = w_sub(w_mul(U00, U11), w_mul(U01, U10))
    # det is a unit (±1, ±ω, ±ω^2)
    det_inv = w_conj(det)   # for units, 1/det = conj(det)
    inv00 = w_mul(det_inv, U11)
    inv01 = w_mul(w_neg(det_inv), U01)
    inv10 = w_mul(w_neg(det_inv), U10)
    inv11 = w_mul(det_inv, U00)
    return (inv00, inv01, inv10, inv11)

# ---------- Decomposition ----------
def decompose_form(a_in, b_in, c_in):
    a, c = a_in, c_in
    b = b_in
    comps = []
    while not (a == 0 and b == (0,0) and c == 0):
        a_red, b_red, c_red, U = reduce_form(a, b, c)
        U_inv = inverse_unimodular(U)
        if a_red == 0:
            k = c_red
            if k == 0:
                break
            # v = (U^{-1})^* e2 = conj(second row of U^{-1})
            v = (w_conj(U_inv[2]), w_conj(U_inv[3]))
            comps.extend([v] * k)
            vp_norm = w_norm(v[0]); vq_norm = w_norm(v[1])
            cross = w_mul(v[0], w_conj(v[1]))
            a -= k * vp_norm
            b = (b[0] - k * cross[0], b[1] - k * cross[1])
            c -= k * vq_norm
        else:
            # v = (U^{-1})^* e1 = conj(first row of U^{-1})
            v = (w_conj(U_inv[0]), w_conj(U_inv[1]))
            comps.append(v)
            vp_norm = w_norm(v[0]); vq_norm = w_norm(v[1])
            cross = w_mul(v[0], w_conj(v[1]))
            a -= vp_norm
            b = (b[0] - cross[0], b[1] - cross[1])
            c -= vq_norm
    return comps

# ---------- Check functions ----------
def check_decomposition(a0, b0, c0, comps):
    ra, rc = 0, 0
    rb = (0,0)
    for p,q in comps:
        ra += w_norm(p)
        rc += w_norm(q)
        cross = w_mul(p, w_conj(q))
        rb = (rb[0] + cross[0], rb[1] + cross[1])
    return (ra == a0 and rb == b0 and rc == c0)

def is_primitive_vec(p, q):
    u, v = p, q
    while v != (0,0):
        # integer division with rounding (same w_div)
        r = w_sub(u, w_mul(w_div(u, v), v))
        u, v = v, r
    return w_norm(u) == 1   # units have norm 1

# ---------- Random PSD matrix (Eisenstein) ----------
def random_psd(max_coeff=5, nsum=4):
    a, c = 0, 0
    b = (0,0)
    for _ in range(random.randint(1, nsum)):
        p = (random.randint(-max_coeff, max_coeff), random.randint(-max_coeff, max_coeff))
        q = (random.randint(-max_coeff, max_coeff), random.randint(-max_coeff, max_coeff))
        a += w_norm(p)
        c += w_norm(q)
        cross = w_mul(p, w_conj(q))
        b = (b[0] + cross[0], b[1] + cross[1])
    return a, b, c

# ---------- Test ----------
if __name__ == "__main__":
    NUM_TESTS = 200_000   # adjust
    random.seed(42)
    print(f"Testing decomposition for Z[ω] — {NUM_TESTS} random matrices...")
    start = time.time()
    failures = 0
    for i in range(NUM_TESTS):
        a, b, c = random_psd(5, 4)
        try:
            comp = decompose_form(a, b, c)
            if not check_decomposition(a, b, c, comp):
                print(f"\nFAIL reconstruction test {i}")
                failures += 1
                break
            # primitivity check
            if not all(is_primitive_vec(p,q) for p,q in comp):
                print(f"\nFAIL primitive test {i}")
                failures += 1
                break
        except Exception as e:
            print(f"\nFAIL exception test {i}: {e}")
            failures += 1
            break
        if i % 100 == 99:
            elapsed = time.time() - start
            print(f"  {i+1} tests done ({elapsed:.1f}s)", end='\r')
            sys.stdout.flush()
        if i % 20000 == 19999:
            print(f"\n --- {i+1} tests passed ---")
    total = time.time() - start
    if failures == 0:
        print(f"\nAll {NUM_TESTS} tests passed in {total:.1f}s.")
    else:
        print(f"\n{failures} failures detected.")